# Hito 3: Ground truth real con IDMT-SMT-Bass

El objetivo de este notebook es incorporar una referencia nota-a-nota
independiente para evaluar transcripciones automáticas de bajo.

Utilizaremos el dataset **IDMT-SMT-Bass-Single-Track**, que proporciona pares
alineados de:

- audio de bajo eléctrico;
- pitch de cada nota;
- onset;
- offset.

En esta primera etapa utilizaremos el par `001.wav` / `001.xml` y
transformaremos la anotación XML al mismo formato tabular utilizado en
`02_metricas_fidelidad.ipynb`.

La referencia obtenida aquí será independiente de Basic Pitch y podrá utilizarse
posteriormente como *ground truth* para calcular Precision, Recall y F1.

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd
import pretty_midi


DATA_ROOT = Path("data/raw/IDMT_SMT_BASS_SINGLE_TRACKS")

xml_path = DATA_ROOT / "annotation" / "001.xml"
audio_path = DATA_ROOT / "audio" / "001.wav"

print(f"XML:   {xml_path}")
print(f"Audio: {audio_path}")
print()
print(f"XML exists:   {xml_path.exists()}")
print(f"Audio exists: {audio_path.exists()}")

XML:   data/raw/IDMT_SMT_BASS_SINGLE_TRACKS/annotation/001.xml
Audio: data/raw/IDMT_SMT_BASS_SINGLE_TRACKS/audio/001.wav

XML exists:   True
Audio exists: True


In [2]:
tree = ET.parse(xml_path)
root = tree.getroot()

global_parameters = root.find("globalParameter")
transcription = root.find("transcription")

print(f"Root tag: {root.tag}")
print()
print(f"Audio declarado en XML: {global_parameters.findtext('audioFileName')}")
print(f"Instrumento:            {global_parameters.findtext('instrument')}")
print(f"Modelo:                  {global_parameters.findtext('instrumentModel')}")
print(f"Número de eventos:       {len(transcription.findall('event'))}")

Root tag: instrumentRecording

Audio declarado en XML: 001.wav
Instrumento:            EBSS
Modelo:                  Fame Baphomet 4 NTB
Número de eventos:       44


In [3]:
events = []

for event in transcription.findall("event"):
    midi = int(event.findtext("pitch"))
    onset_s = float(event.findtext("onsetSec"))
    offset_s = float(event.findtext("offsetSec"))

    events.append(
        {
            "note": pretty_midi.note_number_to_name(midi),
            "midi": midi,
            "onset_s": onset_s,
            "offset_s": offset_s,
            "duration_s": offset_s - onset_s,
            "string_number": int(event.findtext("stringNumber")),
            "fret_number": int(event.findtext("fretNumber")),
            "excitation_style": event.findtext("excitationStyle"),
            "expression_style": event.findtext("expressionStyle"),
        }
    )

reference_001 = pd.DataFrame(events)

reference_001.head(10)

,note,midi,onset_s,offset_s,duration_s,string_number,fret_number,excitation_style,expression_style
0,G2,43,0.00000,0.71717,0.71717,3,5,MU,VI
1,C2,36,0.78263,1.04350,0.26087,2,3,MU,NO
2,D2,38,1.08550,1.32440,0.23890,2,5,MU,NO
3,F2,41,1.32640,1.56530,0.23890,3,3,MU,NO
4,A#1,34,1.58770,2.04090,0.45320,1,6,MU,BE
5,G1,31,2.09850,2.69670,0.59820,1,3,MU,NO
6,C2,36,2.87760,3.13840,0.26080,2,3,MU,NO
7,D2,38,3.17970,3.41190,0.23220,2,5,MU,NO
8,F2,41,3.41190,3.64950,0.23760,3,3,MU,NO
9,G2,43,3.68040,3.90790,0.22750,3,5,MU,NO


## 1. Auditoría del ground truth

Antes de utilizar las anotaciones como referencia para evaluar Basic Pitch,
verificaremos que la conversión desde XML produjo una representación consistente.

Comprobaremos:

- número total de eventos;
- ausencia de valores faltantes;
- que cada offset sea posterior a su onset;
- orden temporal de los eventos;
- rango de pitches MIDI;
- distribución de las duraciones.

Estas comprobaciones permiten detectar errores de parsing antes de utilizar
las anotaciones como *ground truth*.

In [4]:
# Auditoría básica de la anotación 001

n_notes = len(reference_001)
n_missing = int(reference_001.isna().sum().sum())

all_positive_durations = bool(
    (reference_001["duration_s"] > 0).all()
)

onsets_sorted = bool(
    reference_001["onset_s"].is_monotonic_increasing
)

midi_min = int(reference_001["midi"].min())
midi_max = int(reference_001["midi"].max())

print("=== Auditoría reference_001 ===")
print(f"Número de notas:              {n_notes}")
print(f"Valores faltantes:            {n_missing}")
print(f"Duraciones todas positivas:   {all_positive_durations}")
print(f"Onsets ordenados en el tiempo:{onsets_sorted}")
print()
print(f"Pitch MIDI mínimo:            {midi_min}")
print(f"Pitch MIDI máximo:            {midi_max}")

print("\n=== Duración de las notas (s) ===")
display(
    reference_001["duration_s"]
    .describe()
    .to_frame(name="duration_s")
)

=== Auditoría reference_001 ===
Número de notas:              44
Valores faltantes:            0
Duraciones todas positivas:   True
Onsets ordenados en el tiempo:True

Pitch MIDI mínimo:            31
Pitch MIDI máximo:            46

=== Duración de las notas (s) ===


,duration_s
count,44.000000
mean,0.319874
std,0.157990
min,0.128100
25%,0.217775
50%,0.248600
75%,0.443450
max,0.717170


In [5]:
# Distribución de alturas presentes en la anotación

pitch_summary = (
    reference_001
    .groupby(["midi", "note"])
    .size()
    .reset_index(name="count")
    .sort_values("midi")
    .reset_index(drop=True)
)

pitch_summary

,midi,note,count
0,31,G1,4
1,34,A#1,4
2,36,C2,8
3,38,D2,8
4,41,F2,8
5,43,G2,8
6,46,A#2,4


## 2. Auditoría del audio asociado

Antes de utilizar `001.wav` como entrada a Basic Pitch verificaremos sus
propiedades nativas y su compatibilidad temporal con la anotación XML.

Comprobaremos:

- frecuencia de muestreo;
- número de canales;
- resolución en bits;
- número de frames;
- duración total del audio;
- último offset anotado;
- que todas las notas estén contenidas dentro de la duración del archivo.

La inspección se realizará directamente sobre el archivo WAV original,
sin aplicar remuestreo.

In [6]:
import wave

with wave.open(str(audio_path), "rb") as wav_file:
    n_channels = wav_file.getnchannels()
    sample_width_bytes = wav_file.getsampwidth()
    sample_rate_hz = wav_file.getframerate()
    n_frames = wav_file.getnframes()

audio_duration_s = n_frames / sample_rate_hz

first_onset_s = float(reference_001["onset_s"].min())
last_offset_s = float(reference_001["offset_s"].max())

annotations_inside_audio = bool(
    (first_onset_s >= 0)
    and (last_offset_s <= audio_duration_s)
)

print("=== Auditoría 001.wav ===")
print(f"Canales:                      {n_channels}")
print(f"Frecuencia de muestreo:       {sample_rate_hz} Hz")
print(f"Resolución:                   {sample_width_bytes * 8} bits")
print(f"Número de frames:             {n_frames}")
print(f"Duración del audio:           {audio_duration_s:.5f} s")
print()
print(f"Primer onset anotado:         {first_onset_s:.5f} s")
print(f"Último offset anotado:        {last_offset_s:.5f} s")
print(
    f"Margen después última nota:   "
    f"{audio_duration_s - last_offset_s:.5f} s"
)
print()
print(f"Anotaciones dentro del audio: {annotations_inside_audio}")

=== Auditoría 001.wav ===
Canales:                      1
Frecuencia de muestreo:       44100 Hz
Resolución:                   16 bits
Número de frames:             736279
Duración del audio:           16.69567 s

Primer onset anotado:         0.00000 s
Último offset anotado:        16.70310 s
Margen después última nota:   -0.00743 s

Anotaciones dentro del audio: False


## 3. Compatibilidad temporal del dataset completo

En el par `001.wav` / `001.xml`, el último offset anotado excede la duración
del archivo WAV en aproximadamente 7 ms.

Antes de interpretar esta diferencia como un problema, auditaremos los 17 pares
del dataset para determinar si corresponde a una característica sistemática de
las anotaciones.

No se modificarán ni recortarán las anotaciones originales.

In [7]:
audit_rows = []

annotation_dir = DATA_ROOT / "annotation"
audio_dir = DATA_ROOT / "audio"

for xml_file in sorted(annotation_dir.glob("*.xml")):
    track_id = xml_file.stem
    wav_file = audio_dir / f"{track_id}.wav"

    # Leer anotación XML
    tree_i = ET.parse(xml_file)
    root_i = tree_i.getroot()
    transcription_i = root_i.find("transcription")

    events_i = transcription_i.findall("event")

    onsets = [
        float(event.findtext("onsetSec"))
        for event in events_i
    ]

    offsets = [
        float(event.findtext("offsetSec"))
        for event in events_i
    ]

    # Leer cabecera WAV sin remuestreo
    with wave.open(str(wav_file), "rb") as wf:
        sr_i = wf.getframerate()
        n_frames_i = wf.getnframes()
        n_channels_i = wf.getnchannels()

    duration_i = n_frames_i / sr_i
    last_offset_i = max(offsets)

    audit_rows.append(
        {
            "track": track_id,
            "n_notes": len(events_i),
            "sample_rate_hz": sr_i,
            "channels": n_channels_i,
            "audio_duration_s": duration_i,
            "last_offset_s": last_offset_i,
            "end_difference_ms": (
                last_offset_i - duration_i
            ) * 1000,
        }
    )

dataset_audit = pd.DataFrame(audit_rows)

dataset_audit

,track,n_notes,sample_rate_hz,channels,audio_duration_s,last_offset_s,end_difference_ms
0,001,44,44100,1,16.695669,16.7031,7.431066
1,002,48,44100,1,19.200000,19.2000,0.000000
2,003,70,44100,1,20.869569,20.8696,0.030839
3,004,56,44100,1,16.695669,16.6956,-0.068934
4,005,66,44100,1,18.640794,18.6409,0.106349
5,006,44,44100,1,14.769252,14.7692,-0.051701
6,007,56,44100,1,20.210544,20.2104,-0.144218
7,008,68,44100,1,29.538481,29.5384,-0.080726
8,009,40,44100,1,15.360000,15.3600,0.000000
9,010,68,44100,1,26.482789,26.4828,0.010884


In [8]:
dataset_audit["annotation_overrun_ms"] = (
    (dataset_audit["last_offset_s"] - dataset_audit["audio_duration_s"])
    .clip(lower=0)
    * 1000
)

dataset_audit["trailing_audio_ms"] = (
    (dataset_audit["audio_duration_s"] - dataset_audit["last_offset_s"])
    .clip(lower=0)
    * 1000
)

dataset_audit["overrun_within_50ms"] = (
    dataset_audit["annotation_overrun_ms"] <= 50
)

display(
    dataset_audit[
        [
            "track",
            "n_notes",
            "sample_rate_hz",
            "audio_duration_s",
            "last_offset_s",
            "annotation_overrun_ms",
            "trailing_audio_ms",
            "overrun_within_50ms",
        ]
    ]
)

,track,n_notes,sample_rate_hz,audio_duration_s,last_offset_s,annotation_overrun_ms,trailing_audio_ms,overrun_within_50ms
0,001,44,44100,16.695669,16.7031,7.431066,0.000000,True
1,002,48,44100,19.200000,19.2000,0.000000,0.000000,True
2,003,70,44100,20.869569,20.8696,0.030839,0.000000,True
3,004,56,44100,16.695669,16.6956,0.000000,0.068934,True
4,005,66,44100,18.640794,18.6409,0.106349,0.000000,True
5,006,44,44100,14.769252,14.7692,0.000000,0.051701,True
6,007,56,44100,20.210544,20.2104,0.000000,0.144218,True
7,008,68,44100,29.538481,29.5384,0.000000,0.080726,True
8,009,40,44100,15.360000,15.3600,0.000000,0.000000,True
9,010,68,44100,26.482789,26.4828,0.010884,0.000000,True


In [9]:
print("=== Compatibilidad temporal IDMT ===")
print(f"Pares auditados: {len(dataset_audit)}")

print(
    f"Máximo exceso de anotación sobre el WAV: "
    f"{dataset_audit['annotation_overrun_ms'].max():.3f} ms"
)

print(
    f"Máximo audio posterior a la última nota: "
    f"{dataset_audit['trailing_audio_ms'].max():.3f} ms"
)

print(
    f"Pares con exceso > 50 ms: "
    f"{(dataset_audit['annotation_overrun_ms'] > 50).sum()}"
)

print(
    f"Todos compatibles con tolerancia de 50 ms: "
    f"{dataset_audit['overrun_within_50ms'].all()}"
)

=== Compatibilidad temporal IDMT ===
Pares auditados: 17
Máximo exceso de anotación sobre el WAV: 7.431 ms
Máximo audio posterior a la última nota: 997.844 ms
Pares con exceso > 50 ms: 0
Todos compatibles con tolerancia de 50 ms: True


### Resultado de la auditoría temporal

Los 17 pares audio–anotación fueron considerados temporalmente compatibles
para el benchmark.

El mayor exceso de un offset anotado respecto de la duración del archivo WAV
fue de aproximadamente **7.4 ms**, valor muy inferior a la tolerancia mínima
de **50 ms** que se utilizará posteriormente para evaluar offsets.

Algunas grabaciones contienen audio posterior a la última nota anotada. El caso
más evidente es el track `017`, con aproximadamente **1 s** de audio posterior
a la última anotación.

No se modificaron ni recortaron las anotaciones originales.

## 4. Baseline de Basic Pitch sobre `001.wav`

Como primera condición experimental evaluaremos Basic Pitch directamente sobre
el audio de bajo aislado, sin aplicar Demucs ni procesamiento DSP.

La inferencia se realizó con:

- Basic Pitch 0.4.0;
- backend CoreML;
- parámetros por defecto;
- audio original `001.wav`.

Basic Pitch produjo un archivo MIDI y un archivo CSV de eventos. Para la
evaluación utilizaremos las columnas de onset, offset y pitch MIDI del CSV.
El campo `pitch_bend` no será utilizado en este primer benchmark.

In [11]:
import csv

basic_pitch_csv = Path(
    "data/interim/basic_pitch/baseline_001/001_basic_pitch.csv"
)

estimated_events = []

with open(basic_pitch_csv, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)

    header = next(reader)

    for row in reader:
        estimated_events.append(
            {
                "onset_s": float(row[0]),
                "offset_s": float(row[1]),
                "midi": int(row[2]),
                "velocity": int(row[3]),
            }
        )

estimated_001 = pd.DataFrame(estimated_events)

estimated_001["note"] = estimated_001["midi"].apply(
    pretty_midi.note_number_to_name
)

estimated_001["duration_s"] = (
    estimated_001["offset_s"] - estimated_001["onset_s"]
)

estimated_001 = (
    estimated_001
    .sort_values("onset_s")
    .reset_index(drop=True)
)

estimated_001.head(10)

,onset_s,offset_s,midi,velocity,note,duration_s
0,0.011610,0.731429,43,100,G2,0.719819
1,0.011610,0.267029,55,57,G3,0.255420
2,0.801088,1.056508,36,85,C2,0.255420
3,0.801088,0.975238,48,48,C3,0.174150
4,1.091338,1.358367,38,83,D2,0.267029
5,1.323537,1.602177,41,86,F2,0.278639
6,1.323537,1.497687,53,58,F3,0.174150
7,1.323537,1.462857,65,52,F4,0.139320
8,1.590567,1.787937,34,84,A#1,0.197370
9,1.602177,1.764717,46,59,A#2,0.162540


In [12]:
print("=== Basic Pitch baseline 001 ===")
print(f"Notas ground truth: {len(reference_001)}")
print(f"Notas estimadas:    {len(estimated_001)}")
print()

print(
    f"Rango MIDI ground truth: "
    f"{reference_001['midi'].min()}–{reference_001['midi'].max()}"
)

print(
    f"Rango MIDI estimado:      "
    f"{estimated_001['midi'].min()}–{estimated_001['midi'].max()}"
)

print()
print(
    f"Primer onset estimado:     "
    f"{estimated_001['onset_s'].min():.5f} s"
)

print(
    f"Último offset estimado:    "
    f"{estimated_001['offset_s'].max():.5f} s"
)

print(
    f"Duraciones positivas:      "
    f"{(estimated_001['duration_s'] > 0).all()}"
)

outside_pitch_range = estimated_001[
    (estimated_001["midi"] < reference_001["midi"].min())
    | (estimated_001["midi"] > reference_001["midi"].max())
]

print()
print(
    f"Notas estimadas fuera del rango MIDI del ground truth: "
    f"{len(outside_pitch_range)}"
)

=== Basic Pitch baseline 001 ===
Notas ground truth: 44
Notas estimadas:    94

Rango MIDI ground truth: 31–46
Rango MIDI estimado:      28–70

Primer onset estimado:     0.01161 s
Último offset estimado:    16.55449 s
Duraciones positivas:      True

Notas estimadas fuera del rango MIDI del ground truth: 37


In [13]:
estimated_pitch_summary = (
    estimated_001
    .groupby(["midi", "note"])
    .size()
    .reset_index(name="count")
    .sort_values("midi")
    .reset_index(drop=True)
)

estimated_pitch_summary

,midi,note,count
0,28,E1,4
1,31,G1,5
2,33,A1,1
3,34,A#1,4
4,35,B1,3
5,36,C2,7
6,38,D2,8
7,41,F2,8
8,43,G2,12
9,46,A#2,9


## 5. Evaluación cuantitativa del baseline

La transcripción generada por Basic Pitch contiene más eventos que el
*ground truth* y presenta varias detecciones en octavas superiores de las
alturas anotadas.

Sin embargo, una nota no puede clasificarse como correcta o incorrecta
considerando únicamente su pitch. La evaluación debe considerar también
su posición temporal.

Compararemos por tanto las notas estimadas con las 44 notas de referencia
utilizando dos criterios:

1. **pitch + onset**;
2. **pitch + onset + offset**.

Usaremos una tolerancia de 50 ms para el onset y 50 cents para el pitch.
Para el offset se utilizará una tolerancia relativa del 20 % de la duración
de la nota, con un mínimo de 50 ms.

In [14]:
import numpy as np
import mir_eval


def midi_to_hz(midi_notes):
    midi_notes = np.asarray(midi_notes, dtype=float)
    return 440.0 * 2 ** ((midi_notes - 69.0) / 12.0)


# Ground truth
ref_intervals = reference_001[
    ["onset_s", "offset_s"]
].to_numpy(dtype=float)

ref_pitches_hz = midi_to_hz(
    reference_001["midi"]
)

# Basic Pitch
est_intervals = estimated_001[
    ["onset_s", "offset_s"]
].to_numpy(dtype=float)

est_pitches_hz = midi_to_hz(
    estimated_001["midi"]
)

# Pitch + onset
p_onset, r_onset, f1_onset, overlap_onset = (
    mir_eval.transcription.precision_recall_f1_overlap(
        ref_intervals,
        ref_pitches_hz,
        est_intervals,
        est_pitches_hz,
        onset_tolerance=0.050,
        pitch_tolerance=50.0,
        offset_ratio=None,
    )
)

# Pitch + onset + offset
p_offset, r_offset, f1_offset, overlap_offset = (
    mir_eval.transcription.precision_recall_f1_overlap(
        ref_intervals,
        ref_pitches_hz,
        est_intervals,
        est_pitches_hz,
        onset_tolerance=0.050,
        pitch_tolerance=50.0,
        offset_ratio=0.2,
        offset_min_tolerance=0.050,
    )
)

baseline_metrics_001 = pd.DataFrame(
    {
        "criterion": [
            "pitch + onset",
            "pitch + onset + offset",
        ],
        "precision": [
            p_onset,
            p_offset,
        ],
        "recall": [
            r_onset,
            r_offset,
        ],
        "f1": [
            f1_onset,
            f1_offset,
        ],
        "avg_overlap_ratio": [
            overlap_onset,
            overlap_offset,
        ],
    }
)

baseline_metrics_001

,criterion,precision,recall,f1,avg_overlap_ratio
0,pitch + onset,0.436170,0.931818,0.594203,0.797955
1,pitch + onset + offset,0.351064,0.750000,0.478261,0.867803


In [15]:
n_reference = len(reference_001)
n_estimated = len(estimated_001)

# Los TP se reconstruyen a partir del recall y el número
# de eventos de referencia.
tp_onset = round(r_onset * n_reference)
fp_onset = n_estimated - tp_onset
fn_onset = n_reference - tp_onset

tp_offset = round(r_offset * n_reference)
fp_offset = n_estimated - tp_offset
fn_offset = n_reference - tp_offset

baseline_counts_001 = pd.DataFrame(
    {
        "criterion": [
            "pitch + onset",
            "pitch + onset + offset",
        ],
        "TP": [tp_onset, tp_offset],
        "FP": [fp_onset, fp_offset],
        "FN": [fn_onset, fn_offset],
        "n_reference": [n_reference, n_reference],
        "n_estimated": [n_estimated, n_estimated],
    }
)

baseline_counts_001

,criterion,TP,FP,FN,n_reference,n_estimated
0,pitch + onset,41,53,3,44,94
1,pitch + onset + offset,33,61,11,44,94


In [16]:
# Matching explícito para el criterio pitch + onset
matching_onset = mir_eval.transcription.match_notes(
    ref_intervals,
    ref_pitches_hz,
    est_intervals,
    est_pitches_hz,
    onset_tolerance=0.050,
    pitch_tolerance=50.0,
    offset_ratio=None,
)

matched_ref_idx = {ref_idx for ref_idx, est_idx in matching_onset}
matched_est_idx = {est_idx for ref_idx, est_idx in matching_onset}

false_positive_idx = [
    idx for idx in range(len(estimated_001))
    if idx not in matched_est_idx
]

false_negative_idx = [
    idx for idx in range(len(reference_001))
    if idx not in matched_ref_idx
]

false_positives_001 = (
    estimated_001.loc[false_positive_idx]
    .sort_values("onset_s")
    .reset_index(drop=True)
)

false_negatives_001 = (
    reference_001.loc[false_negative_idx]
    .sort_values("onset_s")
    .reset_index(drop=True)
)

print(f"Matches / TP: {len(matching_onset)}")
print(f"False positives: {len(false_positives_001)}")
print(f"False negatives: {len(false_negatives_001)}")

Matches / TP: 41
False positives: 53
False negatives: 3


In [17]:
false_negatives_001[
    ["note", "midi", "onset_s", "offset_s", "duration_s"]
]

,note,midi,onset_s,offset_s,duration_s
0,C2,36,7.0408,7.1716,0.1308
1,G1,31,14.6445,15.2464,0.6019
2,C2,36,15.3879,15.5160,0.1281


In [18]:
fp_diagnosis = []

for _, fp in false_positives_001.iterrows():

    # Referencias cuyo onset ocurre dentro de ±50 ms
    nearby_ref = reference_001[
        (reference_001["onset_s"] - fp["onset_s"]).abs() <= 0.050
    ].copy()

    if len(nearby_ref) == 0:
        fp_diagnosis.append(
            {
                "onset_s": fp["onset_s"],
                "midi": fp["midi"],
                "note": fp["note"],
                "nearest_ref_note": None,
                "semitone_difference": None,
                "onset_error_ms": None,
                "category": "no nearby reference onset",
            }
        )
        continue

    nearby_ref["onset_error_s"] = (
        nearby_ref["onset_s"] - fp["onset_s"]
    ).abs()

    nearest = nearby_ref.loc[
        nearby_ref["onset_error_s"].idxmin()
    ]

    semitone_difference = int(
        fp["midi"] - nearest["midi"]
    )

    if semitone_difference in (12, 24, 36):
        category = "upper octave/harmonic"
    elif semitone_difference in (-12, -24, -36):
        category = "lower octave"
    else:
        category = "other nearby pitch"

    fp_diagnosis.append(
        {
            "onset_s": fp["onset_s"],
            "midi": fp["midi"],
            "note": fp["note"],
            "nearest_ref_note": nearest["note"],
            "semitone_difference": semitone_difference,
            "onset_error_ms": 1000 * nearest["onset_error_s"],
            "category": category,
        }
    )

fp_diagnosis = pd.DataFrame(fp_diagnosis)

fp_diagnosis.head(20)

,onset_s,midi,note,nearest_ref_note,semitone_difference,onset_error_ms,category
0,0.011610,55,G3,G2,12.0,11.609977,upper octave/harmonic
1,0.801088,48,C3,C2,12.0,18.458435,upper octave/harmonic
2,1.323537,53,F3,F2,12.0,2.862585,upper octave/harmonic
3,1.323537,65,F4,F2,24.0,2.862585,upper octave/harmonic
4,1.602177,46,A#2,A#1,12.0,14.476871,upper octave/harmonic
5,1.787937,35,B1,None,NaN,NaN,no nearby reference onset
6,2.125910,43,G2,G1,12.0,27.409751,upper octave/harmonic
7,2.880558,48,C3,C2,12.0,2.958277,upper octave/harmonic
8,3.182418,50,D3,D2,12.0,2.717687,upper octave/harmonic
9,3.426227,53,F3,F2,12.0,14.327211,upper octave/harmonic


In [19]:
fp_category_summary = (
    fp_diagnosis["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)

fp_category_summary["percentage"] = (
    100
    * fp_category_summary["count"]
    / len(fp_diagnosis)
)

fp_category_summary

,category,count,percentage
0,upper octave/harmonic,39,73.584906
1,no nearby reference onset,13,24.528302
2,other nearby pitch,1,1.886792


In [20]:
semitone_summary = (
    fp_diagnosis
    .dropna(subset=["semitone_difference"])
    ["semitone_difference"]
    .value_counts()
    .sort_index()
    .rename_axis("semitone_difference")
    .reset_index(name="count")
)

semitone_summary["percentage_of_all_fp"] = (
    100 * semitone_summary["count"] / len(fp_diagnosis)
)

semitone_summary

,semitone_difference,count,percentage_of_all_fp
0,-15.0,1,1.886792
1,12.0,32,60.377358
2,24.0,7,13.207547


### Diagnóstico de los falsos positivos

El baseline produjo 53 falsos positivos bajo el criterio de pitch + onset.

De ellos:

- **39 (73.6 %)** ocurrieron dentro de ±50 ms del onset de una nota de
  referencia y correspondieron a una o más octavas por encima;
- **13 (24.5 %)** no presentaron un onset de referencia cercano;
- **1 (1.9 %)** correspondió a otro tipo de discrepancia de pitch.

Dentro del total de falsos positivos:

- **32 (60.4 %)** estaban exactamente una octava por encima (+12 semitonos);
- **7 (13.2 %)** estaban dos octavas por encima (+24 semitonos).

Estos resultados son compatibles con una fuerte estructura de error por
octava superior. Aunque una posible explicación acústica es la influencia de
componentes armónicos del bajo, esa interpretación deberá contrastarse antes
de atribuir causalidad.

Por tanto, el primer experimento DSP se diseñará para estudiar si una
modificación controlada del balance espectral puede reducir estas detecciones
adicionales sin deteriorar el elevado Recall del baseline.

## 6. Conclusión del Hito 3

Se incorporó un ground truth independiente utilizando
IDMT-SMT-Bass-Single-Track y se estableció un primer baseline cuantitativo
sobre una grabación real de bajo aislado.

Para `001.wav`, Basic Pitch detectó 94 eventos frente a 44 notas de referencia.

Bajo el criterio pitch + onset:

- TP = 41
- FP = 53
- FN = 3
- Precision = 0.436
- Recall = 0.932
- F1 = 0.594

Al exigir además un offset correcto, el F1 disminuyó a 0.478.

El análisis de los falsos positivos mostró que 39 de 53 (73.6 %) se encontraban
temporalmente asociados a una nota verdadera pero una o dos octavas por encima.
Por tanto, el siguiente experimento estudiará si un procesamiento espectral
controlado puede reducir esta sobre-detección manteniendo el alto Recall del
baseline.

No se modificaron las anotaciones originales ni los parámetros por defecto
de Basic Pitch en esta condición baseline.